# 板块六：张量求导（Autograd）—— 自动微分系统

## 一、基础用法：requires_grad 与 .backward()

### 步骤1：标记需要梯度的张量

In [2]:
import torch

# 创建张量并启用梯度追踪
x = torch.tensor(2.0, requires_grad=True)
print("x.requires_grad:", x.requires_grad)  # True

x.requires_grad: True


### 步骤2：进行前向计算（构建计算图）

In [3]:
y = x**3 + x ** 2 + 3 * x + 1  # y = x² + 3x + 1
print("y:", y)           # tensor(11., grad_fn=<AddBackward0>)

y: tensor(19., grad_fn=<AddBackward0>)


### 步骤3：反向传播（自动求导）

In [ ]:
y.backward()  # 计算 dy/dx
print("dy/dx at x=2:", x.grad)  # tensor(7.)

dy/dx at x=2: tensor(19.)


## 二、多变量与向量求导

### 场景1：多个输入

In [5]:
a = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(3.0, requires_grad=True)

z = a ** 2 + b ** 3
z.backward()

print("dz/da:", a.grad)   # 4.0 (2*a)
print("dz/db:", b.grad)  # 27.0 (3*b²)

dz/da: tensor(4.)
dz/db: tensor(27.)


### 场景2：向量输入（Jacobian 向量积）

In [6]:
x = torch.tensor([2.0, 3.0], requires_grad=True)
y = x ** 2  # [4., 9.]

#  y.backward() 会报错！因为 y 不是标量

#  正确做法：传入 gradient（通常用于链式法则）
y.backward(torch.tensor([1.0, 1.0]))  # 相当于 dL/dy = [1,1]

print("dy/dx:", x.grad)  # [4., 6.] → 即 [2*2, 2*3]

dy/dx: tensor([4., 6.])


## 三、torch.no_grad()：禁用梯度计算
在推理（inference）或评估阶段，不需要梯度，关闭可节省内存和加速

In [7]:
model = torch.nn.Linear(10, 1)
x = torch.randn(5, 10)

# 训练模式：需要梯度
output_train = model(x)  # grad_fn 存在

# 推理模式：禁用梯度
with torch.no_grad():
    output_eval = model(x)  # grad_fn=None，不追踪

print("Train has grad_fn:", output_train.grad_fn is not None)  # True
print("Eval has grad_fn:", output_eval.grad_fn is not None)    # False


Train has grad_fn: True
Eval has grad_fn: False


## 四、梯度清零：optimizer.zero_grad() 或 x.grad.zero_()
梯度是累加的！每次反向传播前必须清零，否则会错误累积。

In [8]:
x = torch.tensor(2.0, requires_grad=True)

for i in range(3):
    y = x ** 2
    y.backward()
    print(f"Iter {i}: grad = {x.grad}")  # 第一次 4，第二次 8，第三次 12！
    # 必须手动清零
    x.grad.zero_()

Iter 0: grad = 4.0
Iter 1: grad = 4.0
Iter 2: grad = 4.0


In [9]:
# 以下是在训练循环中的标准写法
# optimizer.zero_grad()  # 清零所有参数梯度
# loss.backward()       # 反向传播
# optimizer.step()      # 更新参数

## 动手练习：线性回归的手动训练

In [10]:
# 数据
x = torch.randn(100, 1)
y = 2 * x + 3 + 0.1 * torch.randn(100, 1)  # 真实: y = 2x + 3

# 参数（需梯度）
w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

learning_rate = 0.01

for epoch in range(100):
    # 前向
    y_pred = w * x + b
    loss = ((y_pred - y) ** 2).mean()
    
    # 反向
    loss.backward()
    
    # 更新参数（注意：不使用 optimizer）
    with torch.no_grad():
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad
        # 清零梯度
        w.grad.zero_()
        b.grad.zero_()
    
    if epoch % 20 == 0:
        print(f"Epoch {epoch}: loss={loss.item():.4f}, w={w.item():.2f}, b={b.item():.2f}")

# 最终应接近 w≈2.0, b≈3.0

Epoch 0: loss=13.9946, w=0.05, b=0.06
Epoch 20: loss=5.8654, w=0.82, b=1.03
Epoch 40: loss=2.4837, w=1.29, b=1.69
Epoch 60: loss=1.0633, w=1.57, b=2.12
Epoch 80: loss=0.4614, w=1.75, b=2.42


# 总结

| 要点                     | 说明                                               |
|--------------------------|----------------------------------------------------|
| `requires_grad=True`     | 标记需要追踪梯度的张量                             |
| `.backward()`            | 从标量损失出发，反向计算所有梯度                   |
| 梯度累加                 | 每次 `backward()` 会累加到 `.grad`，必须清零       |
| `torch.no_grad()`        | 推理时禁用梯度，提升效率                           |
| 仅标量可直接 `backward()`| 向量需传 `gradient` 参数                           |